# Computational Creativity Assessment

'Creativity' is not solitely computationally detectable, since there is no binary or selective way of 'detecting' creativity. There is, however, research done on the use of several computational proxies for creativity (e.g. Kyle et al., 2018; Crossley et al., 2016; Bojanowski et al., 2017). In this section, a selection of these approaches will be used to computationally approach creativity measures in texts. 

Chakrabarty et al. (2024)  contucted a survey among 10 creative writers to evaluate 48 human-written stories. The feedback of the writers was transfomed into a 'creativity measure test', the Torrance Test of Creative Writing (TTCW, inspired on the Torrance Test of Creative Thinking (Torrance, 1966)). This test is used in the current study to collect human feedback on both human-written texts and LLM-written texts. 

The section will be stuctured using the 14 questions within the 4 categories of the TTCW. Each question will be approached by a different computational method. In the end, a 'general computational creativity score' will be calculated using the results of all the methods.  

## Imports 

Make sure to have 'nl_core_news_lg' installed seperately 

In [ ]:
import pandas as pd
import numpy as np
import spacy
from collections import Counter
from pathlib import Path
import re
nlp = spacy.load("nl_core_news_lg")

## Creating a full dataset with all stories 

In [ ]:
folders = {
    "human": "human_stories",
    "gpt": "GPT_stories",
    "claude": "claude_stories",
    "gemini": "gemini_stories"
}

rows = []

for source, folder in folders.items():

    for file in Path(folder).glob("*.txt"):

        text = file.read_text(encoding="utf-8")

        match = re.search(r'(\d+)', file.stem)
        story_id = int(match.group(1)) if match else None

        rows.append({
            "source": source,
            "story_id": story_id,
            "filename": file.name,
            "text": text
        })

df = pd.DataFrame(rows)

df = df.sort_values(["source", "story_id"]).reset_index(drop=True)

df["prompt_id"] = df["story_id"]

print(df.head())
print(df.shape)
df.to_csv("creativity_dataset.csv", index=False)

For the purpose of extracting linguistic features, the text will be transformed into a SpaCy Doc. 

In [ ]:
def preprocess(text):
    doc = nlp(text)
    sentences = list(doc.sents)
    tokens = [t.text for t in doc if not t.is_space]
    return doc, sentences, tokens

## Fluency 

### TTCW1 - Narrative pacing  

Narrative pacing refers to the way the writer shapes time in the story. They can either stretch time (so that the time-period seems 'longer' in the story than it would be in the real-world: e.g. heavy descriptions of a split second event) or compress time (so that the time-period seems 'shorter': e.g. summarization of a sequence of events). 'Skilled' writers use time effectively to slow down and speed up to focus on what they think is crucial for the story (Propp, 2026). In this study, narrative pacing will be proxied by (1) Event Density Analysis, (2) Temporal Expression Extraction, (3) Sentence-to-Summary ratio and (4) Sentence Rythm Variation. 

#### 1. Event Density Analysis 

Event Density refers to the number of events described in a vast number of words (e.g. 100). If one story has more events occuring in the same amount of words as another story, one could infer that the time in story 1 is more compressed then the time is story 2. 

What is interesting to look at is the variation of event density across the stroy as a whole. Does the writer use the manipulation of time to tell his story? Or is time used in a 'flat' way? 

In [ ]:
df = pd.read_csv("creativity_dataset.csv")

These events can by proxied by verbs. Main action verbs (e.g. 'rennen', 'vliegen') are more important for indicating events in a story than 'helping' verbs (e.g. 'is', 'was', 'ben') Since the texts are already transformed into a SpaCy Doc, it is easy to extract them using SpaCy's Dutch language model. 

In [ ]:
def extract_events(doc):
    return [
        token
        for token in doc
        if token.dep_ == "ROOT" and token.pos_ == "VERB"
    ]

To extract the variation of event desity across the story, the story should be split into several 'parts'.

In [ ]:
def sliding_event_density(doc, window_size=100, step_size=20):
    tokens = [t for t in doc if not t.is_space]
    
    results = []
    
    for start in range(0, len(tokens) - window_size + 1, step_size):
        window = tokens[start:start + window_size]
        
        word_count = len(window)
        
        event_count = sum(
            1 for t in window
            if t.dep_ == "ROOT" and t.pos_ == "VERB"
        )
        
        density = event_count / word_count if word_count > 0 else 0
        
        results.append(density)
    
    return results

In [ ]:
results = []

for i, row in df.iterrows():
    text = row["text"]
    
    doc = nlp(text)
    
    density_curve = sliding_event_density(doc)
    
    results.append({
        "story_id": row["story_id"],
        "source": row["source"],
        "set": row["set"],
        "prompt_id": row["prompt_id"],
        "density_curve": density_curve
    })

results_df = pd.DataFrame(results)

results_df.head()

Temporal expression density 

In [ ]:
TEMP_WORDS = [
    "later", "then", "suddenly", "meanwhile", "after", "before",
    "gisteren", "vandaag", "toen", "vervolgens", "plots"
]

In [ ]:
def temporal_density(text):
    text_lower = text.lower()
    count = sum(text_lower.count(w) for w in TEMP_WORDS)
    return count / max(len(text.split()), 1) * 100

Scene // Summary ratio

In [ ]:
def sentence_type(sentence):
    s = sentence.text.lower()

    scene_markers = ['"', "'", "said", "looked", "walked", "saw"]
    summary_markers = ["later", "years", "while", "after", "before"]

    scene_score = sum(m in s for m in scene_markers)
    summary_score = sum(m in s for m in summary_markers)

    if scene_score > summary_score:
        return "scene"
    elif summary_score > scene_score:
        return "summary"
    else:
        return "neutral"

In [ ]:
def scene_summary_ratio(sentences):
    labels = [sentence_type(s) for s in sentences]

    summary = labels.count("summary")
    scene = labels.count("scene")

    return scene / max(summary + scene, 1)

Sentence Rythm Variation

In [ ]:
def sentence_rhythm(sentences):
    lengths = [len(s.text.split()) for s in sentences]

    return {
        "mean_len": np.mean(lengths),
        "std_len": np.std(lengths),
        "cv_len": np.std(lengths) / max(np.mean(lengths), 1)
    }

Integrating

In [ ]:
def fluency1_score(text):

    doc, sentences, tokens = preprocess(text)

    events = extract_events(doc)

    ed = event_density(tokens, events)
    rolling_ed = rolling_event_density(doc)
    ed_feats = event_density_features(rolling_ed)

    temp = temporal_density(text)
    ssr = scene_summary_ratio(sentences)
    rhythm = sentence_rhythm(sentences)

    return {
        "event_density": ed,
        **ed_feats,
        "temporal_density": temp,
        "scene_summary_ratio": ssr,
        **rhythm
    }

In [ ]:
results = []

for _, row in df.iterrows():
    feats = fluency1_score(row["text"])
    feats["story_id"] = row["story_id"]
    feats["condition"] = row["condition"]
    results.append(feats)

features_df = pd.DataFrame(results)

Analysis

In [ ]:
features_df.groupby("condition").mean()

In [ ]:
import scipy.stats as stats

stats.ttest_ind(
    features_df[features_df.condition=="human"]["ed_std"],
    features_df[features_df.condition=="gpt"]["ed_std"]
)

In [ ]:
import matplotlib.pyplot as plt

sample = rolling_event_density(nlp(df.iloc[0]["text"]))

plt.plot(sample)
plt.title("Narrative Pacing Curve (Event Density)")
plt.show()